# 202 — Train Sentinel-2 Model

Trains Random Forest on S2 spectral features.
No NDVI purity filter — uses natural class distribution.
Run after `201_prepare_sat.ipynb`.

In [ ]:
from config_s2 import *
from helpers_s2 import *
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

QA_DIR = os.path.join(BASE_DIR, "qa_visuals", "sentinel2")
os.makedirs(QA_DIR, exist_ok=True)
print(f"Config loaded ✓  |  Cores: {N_CORES}")

In [ ]:
# ── Step 1: extract pixels ────────────────────────────────────────────────────
print("── Step 1: Extract training pixels ─────────────────────────")

qa_df  = pd.read_csv(os.path.join(S2_DIR, "s2_mask_qa.csv"))
tiles  = qa_df["tile"].tolist()
print(f"  Tiles: {len(tiles)}")

np.random.seed(RANDOM_SEED)
all_features, all_labels = [], []

for i, fname in enumerate(tiles):
    img_path  = os.path.join(S2_IMG_DIR,  fname)
    mask_path = os.path.join(S2_MASK_DIR, fname)
    if not os.path.exists(img_path) or not os.path.exists(mask_path):
        print(f"  [!] Missing: {fname}")
        continue

    with rasterio.open(img_path)  as src: img  = src.read()
    with rasterio.open(mask_path) as src: mask = src.read(1).reshape(-1)

    features  = engineer_features_s2(img, NODATA_VAL, S2_SCALE)
    valid     = (features != 0).any(axis=1)
    features  = features[valid]
    mask_flat = mask[valid]

    all_features.append(features)
    all_labels.append(mask_flat)

    lupine_px = int((mask_flat == 1).sum())
    print(f"  [{i+1}/{len(tiles)}] {fname}  lupine={lupine_px:,}  total={len(mask_flat):,}")
    del img, features, mask; gc.collect()

X = np.vstack(all_features).astype(np.float32)
y = np.concatenate(all_labels).astype(np.uint8)
print(f"\n  Total: {len(X):,}  Lupine: {y.sum():,} ({y.mean():.1%})  BG: {(y==0).sum():,}")

In [ ]:
# ── Step 2: train RF ─────────────────────────────────────────────────────────
print("── Step 2: Train RF ─────────────────────────────────────────")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=1-TRAIN_SPLIT, random_state=RANDOM_SEED, stratify=y)
print(f"  Train: {len(X_train):,}  Val: {len(X_val):,}")

rf = RandomForestClassifier(
    n_estimators=50, max_depth=15,
    n_jobs=-1, random_state=RANDOM_SEED, verbose=1)
rf.fit(X_train, y_train)
joblib.dump(rf, MODEL_PATH_S2)
print(f"  Model saved → {MODEL_PATH_S2}")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
rf = joblib.load(MODEL_PATH_S2)
rf.n_jobs = -1
print(f"Model: {MODEL_NAME_S2}  |  Trees: {rf.n_estimators}  |  Features: {rf.n_features_in_}")

In [ ]:
# ── Step 3: evaluate ─────────────────────────────────────────────────────────
y_pred = rf.predict(X_val)
print(f"  F1:        {f1_score(y_val, y_pred):.4f}")
print(f"  Precision: {precision_score(y_val, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_val, y_pred):.4f}")
print(f"  Accuracy:  {accuracy_score(y_val, y_pred):.4f}")

cm      = confusion_matrix(y_val, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"S2 model — {MODEL_NAME_S2}", fontsize=11)
sns.heatmap(cm,      annot=True, fmt=",",    cmap="Blues",
            xticklabels=["BG","Lupine"], yticklabels=["BG","Lupine"], ax=axes[0])
sns.heatmap(cm_norm, annot=True, fmt=".3f",  cmap="Blues",
            xticklabels=["BG","Lupine"], yticklabels=["BG","Lupine"], ax=axes[1])
axes[0].set_title("Raw"); axes[1].set_title("Normalised")
plt.tight_layout()
plt.savefig(os.path.join(QA_DIR, f"confusion_{MODEL_NAME_S2}.png"), dpi=150, bbox_inches="tight")
plt.show()

importance = pd.DataFrame({"feature": FEATURE_NAMES_S2, "importance": rf.feature_importances_})
importance = importance.sort_values("importance", ascending=False)
print("\nFeature importance:")
print(importance.to_string(index=False))